# Конспект. Модуль 9: Регуляризация и борьба с переобучением — сводный взгляд

## 1. Зачем это нужно и как это связано с предыдущими модулями

Модули 1–8 вводили регуляризационные механизмы **по одному**, каждый раз в контексте, где он возникал естественно: `max_depth`/`min_samples_leaf` в Модуле 1 (одиночное дерево), `max_features` в Модуле 2 (декорреляция ансамбля), `learning_rate`/`subsample`/ранняя остановка в Модуле 5 (динамика бустинга), `λ`/`γ` в Модуле 6 (регуляризованная целевая функция), `num_leaves`/`min_data_in_leaf`/`lambda_l1`/`lambda_l2`/`min_gain_to_split` в Модуле 7 (LightGBM), `depth`/`l2_leaf_reg` в Модуле 8 (CatBoost).

К этому моменту у вас накопилось **больше десятка** названий параметров из трёх разных библиотек — легко запутаться, какой из них за что отвечает и какой лучше крутить в первую очередь. Этот модуль **не вводит новую математику** — это модуль-синтез: мы группируем всё изученное **не по библиотекам, а по механизму действия** (что именно каждый параметр ограничивает), и осваиваем практический навык — систематическое «лечение» переобученной модели, шаг за шагом.

## 2. Таксономия регуляризации по механизму действия

Вместо того чтобы запоминать параметры библиотека за библиотекой, полезнее мыслить **категориями**: любой регуляризационный параметр бустинга ограничивает **что-то одно** из пяти уровней модели. Разберём каждую категорию, вспоминая, откуда именно взялась логика в предыдущих модулях.

### Категория A — Сложность структуры отдельного дерева

Мы вывели в Модуле 1: чем глубже/детальнее дерево, тем ниже его bias, но выше variance. Все параметры этой категории напрямую ограничивают, **сколько «вопросов»** дерево может задать.

| Библиотека | Параметр | Что ограничивает |
|---|---|---|
| Все | `max_depth` | Максимальная глубина |
| LightGBM | `num_leaves` | Число листьев — **главный** параметр при leaf-wise росте (Модуль 7, раздел 3.3) |
| CatBoost | `depth` | Глубина симметричного дерева — здесь глубина честно определяет сложность (`2^depth` листьев гарантированно, Модуль 8) |
| Все (разные названия) | `min_data_in_leaf` (LightGBM) / `min_child_weight` (XGBoost) / `min_samples_leaf` (sklearn/CatBoost аналог) | Минимальный «вес» объектов в листе — прямой аналог Модуля 1 |
| XGBoost/LightGBM | `min_gain_to_split` / `γ` | Минимальный Gain, оправдывающий разбиение (Модуль 6, формула Gain) |

**Важный нюанс про `min_child_weight` в XGBoost, который часто путают.** Это **не** буквально «минимум объектов в листе», как можно подумать по аналогии с `min_samples_leaf`. Вспомните Модуль 6: у каждого листа есть `H_j = Σ(i∈I_j) h_i` — сумма **гессианов**, а не просто счётчик объектов. `min_child_weight` ограничивает именно `H_j`. Для MSE, где `h_i=1` константа для всех объектов, `H_j` **численно равен** количеству объектов в листе — тогда `min_child_weight` действительно ведёт себя как «минимум объектов». Но для LogLoss, где `h_i=p_i(1-p_i)` (Модуль 6, раздел 5.2) **меняется от объекта к объекту**, `H_j` — это **взвешенная** сумма, где объекты с `p≈0.5` (модель не уверена) вносят больший вклад в `H_j`, чем объекты с `p≈0` или `p≈1` (модель уверена). Практическое следствие: в классификации `min_child_weight` фактически требует, чтобы в листе было **достаточно статистически неопределённых, «спорных»** объектов, а не просто достаточно объектов вообще — лист из 50 объектов, в котором модель уверена (`p` близко к 0 или 1 у всех), может иметь **меньший** `H_j`, чем лист из 20 «спорных» объектов.

### Категория B — Регуляризация значений (весов) в листьях

Прямое продолжение Модуля 6: `w* = -G_j/(H_j+λ)`.

| Библиотека | Параметр | Тип |
|---|---|---|
| XGBoost | `reg_lambda` (`lambda`) | L2 |
| XGBoost | `reg_alpha` (`alpha`) | L1 |
| LightGBM | `lambda_l2` | L2 |
| LightGBM | `lambda_l1` | L1 |
| CatBoost | `l2_leaf_reg` | L2 |

**Эффект (численно продемонстрирован в Модуле 6, раздел 5.1):** увеличение `λ` сжимает `w_j*` к нулю — модель делает менее экстремальные предсказания в каждом отдельном листе, прямая аналогия Ridge-регрессии (Неделя 4), только применённая к предсказаниям листьев, а не к весам линейной модели.

### Категория C — Стохастичность на уровне одной итерации/дерева

Продолжение Модуля 2 (декорреляция через `max_features`) и Модуля 5 (`subsample`), усиленное в Модуле 7 идеей GOSS.

| Библиотека | Параметр (строки) | Параметр (признаки) |
|---|---|---|
| XGBoost | `subsample` | `colsample_bytree` (и его варианты `colsample_bylevel`, `colsample_bynode`) |
| LightGBM | `bagging_fraction` + `bagging_freq` | `feature_fraction` |
| CatBoost | `subsample` (Bernoulli-сэмплирование) | `rsm` (Random Subspace Method) |
| LightGBM (специфично) | `top_rate`, `other_rate` (GOSS, Модуль 7, раздел 4) | — |

### Категория D — Регуляризация на уровне всего ансамбля

Продолжение Модуля 5: контроль общей сложности **аддитивной функции**, а не отдельного дерева.

| Библиотека | Параметр |
|---|---|
| Все | `learning_rate` / `eta` — shrinkage (Модуль 5, раздел 2) |
| Все | `n_estimators` / `iterations` + ранняя остановка (Модуль 5, разделы 4–5) |

### Категория E — Структурная регуляризация (специфична для CatBoost)

Не «параметр для подкручивания», а **архитектурный выбор**, разобранный в Модуле 8: сами Oblivious Trees (раздел 2 Модуля 8) действуют как регуляризатор за счёт структурного ограничения на общий сплит для всего уровня, а выбор `boosting_type='Ordered'` защищает от специфической утечки через переиспользование данных (Модуль 8, раздел 3).

## 3. Сводная таблица параметров трёх библиотек (для быстрой справки)

| Механизм | XGBoost | LightGBM | CatBoost |
|---|---|---|---|
| Глубина/сложность дерева | `max_depth` | `num_leaves`, `max_depth` | `depth` |
| Мин. «веса» в листе | `min_child_weight` (сумма гессианов!) | `min_data_in_leaf` | `min_data_in_leaf` |
| Минимальный Gain для сплита | `gamma` | `min_gain_to_split` | — (регулируется в основном через `l2_leaf_reg` и структуру) |
| L1 на листьях | `reg_alpha` | `lambda_l1` | — |
| L2 на листьях | `reg_lambda` | `lambda_l2` | `l2_leaf_reg` |
| Сэмплирование строк | `subsample` | `bagging_fraction`(+`bagging_freq`) | `subsample` |
| Сэмплирование признаков | `colsample_bytree` | `feature_fraction` | `rsm` |
| Learning rate | `eta` | `learning_rate` | `learning_rate` |
| Число итераций | `n_estimators` | `n_estimators` | `iterations` |

**Практический вывод из этой таблицы:** осваивая **одну** библиотеку глубоко (мы делали это для LightGBM в Модуле 7 и CatBoost в Модуле 8), вы фактически осваиваете **все три** — названия параметров различаются, но категория и математический смысл (из раздела 2) — одни и те же. На собеседовании, если вас спросят про параметр незнакомой вам конкретно библиотеки (скажем, вы работали с LightGBM, а спрашивают про XGBoost), правильная стратегия — не паниковать от незнакомого названия, а спросить себя «к какой из пяти категорий это, вероятно, относится по смыслу названия?» — чаще всего этого достаточно, чтобы дать содержательный ответ.

## 4. Как читать learning curve — полная систематика

В Модуле 5 мы увидели одну характерную картину (val loss падает, потом растёт). Здесь — полная систематика **трёх** паттернов, которые нужно уметь различать мгновенно, плюс два практических осложнения.

### 4.1. Три базовых паттерна

**Недообучение (underfitting):**

In [ ]:
Loss
 │  train ────╲___________________________  (высокий, но стабильный)
 │  val   ────╲___________________________  (высокий, идёт ПОЧТИ ВПЛОТНУЮ к train)
 └──────────────────────────────────────────► итерация

Обе кривые близки друг к другу (небольшой gap), но **обе** останавливаются на высоком уровне ошибки — модели не хватает сложности (слишком маленький `num_leaves`, слишком большой `min_data_in_leaf`, слишком сильная регуляризация), она не в состоянии выучить даже то, что доступно на train.

**Переобучение (overfitting):**

In [ ]:
Loss
 │  train ──────────────────────────╲______________  (продолжает падать)
 │  val   ────╲______________________╱‾‾‾‾‾‾‾‾‾‾‾‾‾  (падает, потом РАСТЁТ)
 └──────────────────────────────────────────► итерация
                                     ▲
                                     точка расхождения (начало переобучения)

Классическая картина Модуля 5: train продолжает улучшаться, val сначала улучшается, затем **разворачивается и ухудшается** — растущий gap между кривыми.

**Оптимум (sweet spot):**

Это не отдельная третья форма кривой, а **конкретная точка** на кривой переобучения — минимум val-кривой, **до** того как она начинает расти. Задача тюнинга — найти конфигурацию гиперпараметров, при которой этот минимум **самый низкий** (а не просто остановиться пораньше на плохой конфигурации).

### 4.2. Осложнение №1: val-кривая выходит на плато, а не явно растёт

Иногда, особенно при уже включённой умеренной регуляризации (Категория C — `subsample`/`feature_fraction`), val-кривая после минимума **не** явно разворачивается вверх, а просто **выходит на плато**, слегка «дрожа» около одного уровня. Это происходит потому, что стохастичность каждого отдельного дерева (регуляризация Категории C) сама по себе смягчает эффект «лишних» деревьев — каждое новое дерево вносит меньший, более случайный вклад, поэтому явного вреда от избыточных итераций не видно так резко, как на полностью нерегуляризованной модели.

**Практическое следствие:** при работе с уже частично регуляризованной моделью параметр `patience` (Модуль 5, `n_iter_no_change`) для ранней остановки может потребоваться **больше**, чем на нерегуляризованной модели — плато может быть длинным, и слишком нетерпеливая остановка (маленький `patience`) рискует остановиться на случайном локальном шуме до того, как val-метрика реально нашла свой истинный минимум.

### 4.3. Осложнение №2: шумная val-кривая

На **маленьких** валидационных выборках сама val-кривая может быть «дёрганой» — скакать вверх-вниз от итерации к итерации не из-за реального переобучения, а просто из-за статистического шума небольшой выборки (случайные колебания метрики на нескольких сотнях объектов). В этом случае искать **единственный точный минимум** по одной «дёрганой» точке ненадёжно — лучше смотреть на **сглаженный тренд** (скользящее среднее по нескольким соседним итерациям) или, если возможно, использовать **несколько** фолдов кросс-валидации (уже знакомая вам Stratified K-Fold, Неделя 4) и усреднять learning curve по фолдам — это даёт более гладкую и надёжную картину, на которой момент разворота виден отчётливее.

## 5. Практика: намеренно переобучаем, затем лечим шаг за шагом

Это центральное практическое упражнение модуля — не просто код для запуска, а **методология** систематического тюнинга: применяем регуляризацию **по одной категории за раз**, в порядке от Категории A (обычно даёт наибольший эффект) к Категории D, отслеживая, как сжимается разрыв train/val на каждом шаге.

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
from sklearn.metrics import log_loss

X, y = make_classification(n_samples=30000, n_features=40, n_informative=15,
                            flip_y=0.08, weights=[0.9, 0.1], random_state=42)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

def evaluate(params, label):
    model = lgb.LGBMClassifier(random_state=42, verbose=-1, **params)
    model.fit(X_train, y_train)
    train_loss = log_loss(y_train, model.predict_proba(X_train)[:, 1])
    val_loss = log_loss(y_val, model.predict_proba(X_val)[:, 1])
    print(f"{label:45s} | train={train_loss:.4f} | val={val_loss:.4f} | gap={val_loss-train_loss:+.4f}")
    return val_loss

# Шаг 0: намеренно переобученная конфигурация
# - огромный num_leaves, отсутствие min_data_in_leaf, отсутствие L1/L2,
#   отсутствие стохастики, крупный learning_rate, много деревьев, без ранней остановки
overfit_params = dict(
    num_leaves=2000, max_depth=-1, min_data_in_leaf=1,
    lambda_l1=0, lambda_l2=0,
    feature_fraction=1.0, bagging_fraction=1.0,
    learning_rate=0.3, n_estimators=500,
)
evaluate(overfit_params, "Шаг 0: намеренно переобучено")

# Шаг 1: лечим Категорию A - ограничиваем сложность дерева
step1 = {**overfit_params, "num_leaves": 31, "min_data_in_leaf": 50}
evaluate(step1, "Шаг 1: + ограничение num_leaves/min_data_in_leaf")

# Шаг 2: добавляем Категорию B - L1/L2 регуляризация листьев
step2 = {**step1, "lambda_l1": 1.0, "lambda_l2": 5.0}
evaluate(step2, "Шаг 2: + L1/L2 на листьях")

# Шаг 3: добавляем Категорию C - стохастичность
step3 = {**step2, "feature_fraction": 0.7, "bagging_fraction": 0.7, "bagging_freq": 1}
evaluate(step3, "Шаг 3: + feature_fraction/bagging_fraction")

# Шаг 4: лечим Категорию D - снижаем learning_rate, компенсируем числом деревьев
step4 = {**step3, "learning_rate": 0.03, "n_estimators": 1500}
evaluate(step4, "Шаг 4: + меньший learning_rate, больше деревьев")

**Что вы должны увидеть, запустив это (качественная картина, а не точные цифры — они будут зависеть от вашей версии библиотеки и случайного зерна):**

- **Шаг 0** — очень низкий (почти нулевой) `train` loss и заметно более высокий `val` loss — большой положительный `gap`, классическое переобучение.
- **Шаг 1** обычно даёт **самое заметное** сокращение `gap` из всех шагов — ограничение структуры дерева (Категория A) чаще всего наиболее «дорогой» источник переобучения в стартовой конфигурации, поэтому лечится в первую очередь и даёт наибольший эффект.
- **Шаги 2–3** дают более скромные, но всё же измеримые дополнительные улучшения `gap`, иногда за счёт небольшого роста `train` loss (это ожидаемо и нормально — модель специально становится немного «менее уверенной» на train ради лучшей обобщающей способности).
- **Шаг 4** может как немного улучшить `val`, так и не сильно изменить картину относительно Шага 3 — эффект `learning_rate` в основном раскрывается **вместе** с ранней остановкой (которую в этом упрощённом примере мы не включаем — добавьте `validation_fraction`+`n_iter_no_change` из Модуля 5 самостоятельно как продолжение упражнения, чтобы увидеть полный эффект).

**Методологический вывод упражнения:** не нужно тюнить всё сразу (это как раз задача автоматического поиска — Модуль 10). Вручную диагностировать и лечить модель эффективнее **по категориям**, начиная с той, что физически ограничивает сложность структуры (Категория A) — часто именно она даёт наибольший «бюджетный» выигрыш за один шаг, прежде чем переходить к более тонким инструментам.

## 6. Частые вопросы на собеседовании

| Вопрос | На что обратить внимание в ответе |
|---|---|
| Как вы обычно подходите к тюнингу гиперпараметров бустинга вручную? | По категориям, от структурной сложности дерева (наибольший эффект) к регуляризации листьев, затем к стохастичности, затем к ансамблевым параметрам (`learning_rate`+число итераций) — не всё сразу |
| Чем `min_child_weight` в XGBoost отличается от `min_samples_leaf` в обычном дереве? | `min_child_weight` ограничивает сумму **гессианов** в листе, а не простое число объектов; для MSE это совпадает с числом объектов (гессиан=1), но для LogLoss — это взвешенная величина, где «уверенные» объекты (`p` близко к 0/1) вносят меньший вклад, чем «спорные» (`p≈0.5`) |
| Как отличить недообучение от переобучения по одному графику? | Недообучение — train и val близки, но обе на высоком уровне ошибки; переобучение — train низкий, val заметно выше и/или растёт после минимума; для окончательной уверенности всегда нужно смотреть на обе кривые вместе, а не на одну |
| Почему val-кривая иногда выходит на плато, а не явно растёт после переобучения? | При включённой стохастической регуляризации (subsample/feature_fraction) вклад каждого лишнего дерева слабее и более случаен — вред от избыточных итераций проявляется мягче, что требует большего `patience` при ранней остановке |
| Почему тюнинг стоит начинать с ограничения сложности дерева, а не с L1/L2? | Сложность структуры дерева (num_leaves/max_depth/depth) обычно является доминирующим источником переобучения в нерегуляризованной модели — её ограничение чаще всего даёт наибольший разовый выигрыш, тогда как L1/L2 обычно вносят более тонкую, дополнительную коррекцию поверх уже разумно ограниченной структуры |

## 7. Чек-поинт — попробуйте ответить без подсказок

1. Назовите по одному аналогу `min_data_in_leaf` в XGBoost и CatBoost.
2. Как отличить на графике «недообучение» от «переобучение» от «оптимума»?
3. К какой из пяти категорий регуляризации (раздел 2) относится `learning_rate`, и почему он не входит в категорию «сложность дерева»?
4. Почему `min_child_weight` в XGBoost — это не то же самое, что «минимальное число объектов в листе», хотя на практике для MSE это совпадает?
5. В упражнении раздела 5 мы применяли регуляризацию в порядке A -> B -> C -> D. Что, скорее всего, произойдёт, если применить их в обратном порядке (сначала D, потом C, потом B, и только в конце A)?

## Ответы для самопроверки

<details>
<summary>Раскрыть после того, как попробуете ответить сами</summary>

1. В XGBoost — `min_child_weight` (с важной оговоркой: это сумма гессианов, а не буквальный счётчик объектов, раздел 2). В CatBoost — `min_data_in_leaf` (то же название и смысл, что в LightGBM).

2. Недообучение: обе кривые (train и val) близки друг к другу, но **обе** останавливаются на высоком уровне ошибки — модели не хватает сложности, чтобы выучить закономерность даже на обучающих данных. Переобучение: train продолжает снижаться (часто до очень низких значений), val сначала снижается, потом явно **растёт** (или как минимум сильно отстаёт от train, образуя растущий разрыв) — модель выучила специфику train-выборки, которая не обобщается. Оптимум — это не отдельная форма кривой, а конкретная **точка** на val-кривой (минимум), до того как она начинает расти — цель тюнинга — сделать этот минимум как можно ниже, а не просто остановиться в произвольной точке.

3. `learning_rate` относится к Категории D — регуляризация на уровне **всего ансамбля**. Он не входит в категорию «сложность дерева», потому что не ограничивает структуру **отдельного** дерева (то, сколько листьев/уровней у него будет) — вместо этого он контролирует, насколько **сильно** вклад каждого уже построенного дерева входит в итоговую сумму `F_M(x) = F_0 + η·h_1 + η·h_2 + ...`, то есть управляет эффективной сложностью **всей аддитивной модели целиком**, а не одного её компонента.

4. `min_child_weight` ограничивает `H_j = Σ(i∈I_j) h_i` — сумму **вторых производных** (гессианов) функции потерь по объектам листа, выведенную в Модуле 6. Для MSE `h_i=1` для всех объектов, поэтому `H_j` численно совпадает с простым числом объектов в листе — отсюда и распространённое (но не всегда верное) представление о параметре как о «минимальном числе объектов». Для LogLoss (и любой другой функции потерь, где гессиан не константа) `h_i` варьируется от объекта к объекту (например, `p(1-p)` — больше для «неуверенных» предсказаний модели), поэтому `H_j` — это взвешенная величина, отражающая не просто количество объектов, а их суммарный вклад в локальную кривизну функции потерь.

5. Скорее всего, эффект окажется заметно **менее выраженным и менее эффективным**. Категория D (`learning_rate`+число итераций) регулирует **амплитуду и число** шагов аддитивной модели, но не трогает то, **насколько сложным** может быть каждый отдельный шаг (дерево) — если структура дерева (Категория A) остаётся неограниченной, каждое отдельное дерево всё ещё может сильно переобучаться на своих собственных остатках (аналогично разделу 5.1 Модуля 4, где мы показали, что «сильное» дерево на первой же итерации может довести train-ошибку почти до нуля). Уменьшение `learning_rate` без ограничения структуры дерева лишь **замедляет** скорость, с которой модель придёт к тому же самому переобученному состоянию — не устраняет коренную причину. Начинать с Категории A эффективнее именно потому, что она устраняет источник избыточной гибкости на уровне каждого отдельного строительного блока ансамбля, прежде чем тонко настраивать то, как эти блоки складываются вместе.

</details>